# Named MolSysMT charge consumption

Bounded consumer qualification for DockingMT #40 and MolSysMT #221. Charge calculation and molecular validity belong to MolSysMT. This does not qualify AutoDock typing, AD4 energies or scientific preparation readiness. The explicit-H methanol coordinates are a declared test pose. The 5X72 fixture uses the already documented stereo engine and authorized property loss. The 1VII control uses explicit AMBER14 and expected total +2 e.

Run with `molsyssuite@uibcdf_3.14` and the clean provider source commit `7894435e748bc55254b6c3d2b63ae82c101e5774`. The helper verifies source bytes against that commit and preserves actual producer version metadata separately. See [the contract](named_partial_charges.md).

In [1]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(root))
from devtools.qualify_named_charges import qualify, save
record = qualify()
save(record)
print(record['interpreter'])
print(record['qualified_provider_commit'])

/home/diego/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python
7894435e748bc55254b6c3d2b63ae82c101e5774


## Observe conserved values and decimal rounding

The calculation report retains the original graph scope. A selected OH projection retains its fractional total. Hydrogen transfers preserve the selected total before export; three-decimal PDBQT values have their own observed total and rounding bound. Single-run timings include first-use costs and are not a benchmark.

In [2]:
import pandas as pd
rows = []
for name, case in record['cases'].items():
    audit = case['audit']
    rows.append({'case': name, 'calculated_atoms': audit['partial_charge_assignment']['n_atoms'], 'prepared_atoms': audit['n_atoms'], 'H_transfers': len(audit['charge_projection']['transfers']), 'prepared_total_e': audit['total_charge'], 'PDBQT_total_e': audit['pdbqt']['total_charge'], 'rounding_difference_e': audit['pdbqt']['rounding_difference'], 'charge_audit': audit['assessment'], 'preparation': case['preparation_assessment']['assessment'], 'Vina_admitted': case['native_vina_input_admitted']})
pd.DataFrame(rows).set_index('case')

,calculated_atoms,prepared_atoms,H_transfers,prepared_total_e,PDBQT_total_e,rounding_difference_e,charge_audit,preparation,Vina_admitted
case,,,,,,,,,
methanol_full,6,3,3,-2.775558e-17,-2.775558e-17,0.000000e+00,consistent,provisional,True
methanol_oh_projection,6,2,0,-1.900006e-01,-1.900000e-01,5.791706e-07,consistent,provisional,True
5x72_p59_flexible,39,25,14,-6.938894e-18,-1.000000e-03,-1.000000e-03,consistent,provisional,True
1vii_amber14,596,364,232,2.000000e+00,1.988000e+00,-1.200000e-02,consistent,provisional,True


## Preserve interpretation limits

The receptor source MolSys contains retained original individual charges. It does not silently acquire the consumer's merged charges. Native H5MSM 0.5 cannot preserve this molecular-mechanics attribution. The complete original report, software, atom correspondence, transfer map and full-precision totals live in preparation/result metadata; PDBQT has a bounded attribution remark. Vina parsing proves interoperability only. The default provisional preparation gate remains in place.

In [3]:
assert all(c['audit']['assessment'] == 'consistent' for c in record['cases'].values())
assert all(c['preparation_assessment']['assessment'] == 'provisional' for c in record['cases'].values())
assert all(c['source_unchanged'] and c['native_vina_input_admitted'] for c in record['cases'].values())
assert abs(record['cases']['1vii_amber14']['audit']['total_charge'] - 2) < 1e-12
assert abs(record['cases']['methanol_oh_projection']['audit']['total_charge']) > .1
for limit in record['limits']:
    print(limit)

AutoDock typing remains heuristic and preparation provisional.
No AD4 scoring or charge-model suitability for an energy function is qualified.
Vina ligand parsing is an interoperability control, not scientific validation.
The existing consumer H transfers remain temporary pending molsysmt#223.
Projected charges were calculated on the original full graph, not recalculated on a fragment.
No H5MSM 0.5 molecular-mechanics attribution roundtrip is claimed.
Single-run timings include first-use costs and are not a benchmark.
Other Python minors are qualified separately in CI.
